# DDoS Attack Detection in Network Traffic

### Cybersecurity Threat Detection — Banking-AI

A Distributed Denial of Service (DDoS) attack happens when multiple systems flood a bank's servers with traffic, making them unavailable to real customers. AI can distinguish between a "busy day" and a "malicious attack."

In this notebook:

## Step 0 — Notebook Header

**Prerequisites:** Basic Python (`pandas`, `numpy`, `matplotlib`), familiarity with `scikit-learn` basics, and basic understanding of cybersecurity and threat detection terminology.

**What You Will Learn:**

After completing this notebook, you will be able to:
- Explain the cybersecurity and threat detection problem and why the chosen classification approach suits this banking workflow.
- Implement an end-to-end classification pipeline on synthetic data and evaluate performance with domain-appropriate metrics.
- Assess whether the model is operationally viable, considering error costs, fairness, and the limitations of synthetic data.

**Estimated time:** 35–45 minutes

**Collection context:** DDoS detection, phishing classification, behavioural biometrics, and endpoint security in banking.

## Step 1 — Banking Problem Introduction

**Why this notebook matters:** Banks rely on digital services. A successful DDoS can stop people from accessing their money. Real-time detection allows for automatic mitigation.

**Input data used:** Packets per second (PPS), Bytes per second (BPS), Number of unique source IPs, and average packet size.

**What we predict:** Traffic State (Normal vs DDoS).

**ML method used:** Decision Tree (highly interpretable and fast for real-time systems).

**Learning goal:** Understand how to use numerical traffic features for security classification.

**Primary stakeholders:** security operations analysts, incident responders, CISOs, and fraud prevention teams.

**Comprehension check:** Before looking at the data, write one sentence describing the core decision this model supports and who benefits from getting it right.

**Domain framing note:** This notebook uses synthetic data for educational purposes. It demonstrates decision-support prototyping, not production-ready deployment.

## Step 2 — Environment Setup

Import libraries and set deterministic seeds for reproducibility.

In [ ]:
# Requires: numpy>=1.24, pandas>=2.0, scikit-learn>=1.3, matplotlib>=3.7, seaborn>=0.13

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

sns.set_theme(style="whitegrid")

RANDOM_SEED = 42
RNG = np.random.default_rng(RANDOM_SEED)
plt.rcParams["figure.figsize"] = (10, 6)

print("Environment ready.")


## Step 3 — Data Creation & Context

Build Synthetic Dataset

We'll simulate 2,000 traffic samples.

In [ ]:
n_samples = 2000

# Normal Traffic: Low PPS, low BPS, fewer unique IPs
normal_pps = RNG.normal(500, 100, n_samples // 2)
normal_unique_ips = RNG.normal(50, 10, n_samples // 2)
normal_packet_size = RNG.normal(1200, 200, n_samples // 2)

# DDoS Traffic: Very high PPS, high unique IPs (botnet), small uniform packet sizes
ddos_pps = RNG.normal(10000, 2000, n_samples // 2)
ddos_unique_ips = RNG.normal(1000, 200, n_samples // 2)
ddos_packet_size = RNG.normal(64, 5, n_samples // 2) # Small packets are common in some attacks

df_normal = pd.DataFrame({
    'pps': normal_pps,
    'unique_ips': normal_unique_ips,
    'avg_packet_size': normal_packet_size,
    'label': 0
})

df_ddos = pd.DataFrame({
    'pps': ddos_pps,
    'unique_ips': ddos_unique_ips,
    'avg_packet_size': ddos_packet_size,
    'label': 1
})

df = pd.concat([df_normal, df_ddos]).sample(frac=1).reset_index(drop=True)
print(f"Dataset generated with {len(df)} traffic samples.")

## Step 4 — Exploratory Data Analysis

EDA

How does traffic volume (PPS) compare between normal and attack states?

In [ ]:
plt.figure(figsize=(10, 5))
sns.kdeplot(data=df, x='pps', hue='label', fill=True)
plt.title('Packets Per Second: Normal vs DDoS')
plt.tight_layout()
plt.show()
plt.close()

**Alt-text:** Visualisation titled "Packets Per Second: Normal vs DDoS". The chart highlights key patterns that inform the modelling approach.

## Step 5 — Feature Engineering

Prepare features for modelling. Domain-meaningful transformations help the model learn patterns aligned with banking practice.

In [ ]:
X = df.drop('label', axis=1)
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_SEED)

## Step 6 — Baseline Establishment

A baseline sets the floor that any useful model must beat. Without it, performance numbers have no context.

In [ ]:
# --- Baseline: always predict the majority class ---
baseline_clf = DummyClassifier(strategy='most_frequent', random_state=RANDOM_SEED)
baseline_clf.fit(X_train, y_train)
baseline_pred = baseline_clf.predict(X_test)
baseline_acc = accuracy_score(y_test, baseline_pred)
print(f"Baseline accuracy (majority-class): {baseline_acc:.3f}")
print(f"Any useful model must beat this {baseline_acc:.3f} floor.")

## Step 7 — Model Training & Evaluation

Train the model and compare performance against the baseline.

**Prediction prompt:** Before viewing the results, what performance do you expect and why?

In [ ]:
clf = DecisionTreeClassifier(max_depth=3)
clf.fit(X_train, y_train)

print("Model trained.")

In [ ]:
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('DDoS Detection Confusion Matrix')
plt.tight_layout()
plt.show()
plt.close()

**Alt-text:** Heatmap titled "DDoS Detection Confusion Matrix". The heatmap reveals correlation strength and direction between variables.

## Step 8 — Interpretability & Explainability

Interpret the model in domain terms. A banking professional should be able to assess whether the model's reasoning is credible.

In [ ]:
plt.figure(figsize=(15, 10))
plot_tree(clf, feature_names=X.columns, class_names=['Normal', 'DDoS'], filled=True)
plt.title('How the Model Decides if Traffic is an Attack')
plt.tight_layout()
plt.show()
plt.close()

**Alt-text:** Visualisation titled "How the Model Decides if Traffic is an Attack". The chart highlights key patterns that inform the modelling approach.

What rules did the AI learn? We can visualize the decision logic.

## Step 9 — Operational Application

Demonstrate how the model output feeds into a real banking workflow decision.

In [ ]:
# --- Scenario: score representative cases from the test set ---
print("Operational demonstration — model decision support:\n")
proba = clf.predict_proba(X_test)[:, 1]
low_idx  = int(np.argmin(proba))
high_idx = int(np.argmax(proba))
mid_idx  = int(np.argsort(proba)[len(proba) // 2])

for label, idx in [("Low-risk", low_idx), ("Medium-risk", mid_idx), ("High-risk", high_idx)]:
    row = X_test.iloc[idx]
    prob = proba[idx]
    print(f"{label} case  predicted probability: {prob:.1%}")
    print(f"  Features: {dict(row.round(2))}")
    print()

print("A decision-maker would combine these scores with business rules and domain judgement.")

## Step 10 — Summary & Reflection

**What you accomplished:** You built an end-to-end cybersecurity and threat detection workflow — from problem framing through data creation, baseline comparison, model training, evaluation, and operational demonstration.

**Limitations:**

- The dataset is synthetic and cannot capture the full complexity of real banking data.
- Model performance on synthetic data does not guarantee equivalent performance in production.
- This notebook is an educational prototype. Real deployment requires validated data, regulatory review, and ongoing monitoring.

**Ethical reflection:**

- Threat detection models must balance security effectiveness with employee and customer privacy.
- False positives in security alerts cause alert fatigue and reduce team effectiveness.
- Behavioural biometrics raise consent and surveillance concerns that require clear policies.

**Reflection questions:**

1. What additional data sources or features would you need before trusting this model in a live banking environment?
2. How would the relative cost of false positives versus false negatives change the metric you optimise for?
3. How could you adapt this workflow to a related problem in cybersecurity and threat detection?

## References

- Scikit-learn documentation: [https://scikit-learn.org/stable/](https://scikit-learn.org/stable/)
- Project quality baseline: `QUALITY_STANDARD.md`
- Domain context drawn from public banking and financial services literature.